# From Text to Insight
## Data Analytics and Linguistic Interpretation of *Stranger Things 5* YouTube Comments

End-to-end NLP and Data Analytics pipeline covering:

- data loading and validation;
- linguistic preprocessing;
- exploratory data analysis;
- Transformer-based sentiment analysis;
- TF-IDF + Logistic Regression;
- feature interpretation;
- K-Means clustering;
- Elbow and Silhouette analysis;
- PCA visualisation.

**Important:** the notebook is designed to reproduce the analytical workflow described in the project documentation. Because the original raw YouTube dataset is not included in the repository, the data-loading cell expects a local CSV file supplied by the user.


## 0. Environment setup

In [ ]:
# If running in Google Colab, uncomment the installation commands as needed.

# !pip install -q pandas numpy spacy transformers torch scikit-learn matplotlib
# !python -m spacy download it_core_news_lg

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 1. Load the dataset

In [ ]:
# Expected repository location:
# data/raw/comments.csv
#
# Adapt DATA_PATH if your local file has a different name or format.

DATA_PATH = Path("../data/raw/comments.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place the raw CSV in data/raw/ or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:", list(df.columns))


### 1.1 Identify the comment-text column

The original project uses the comment text as the main analytical variable. The helper below looks for common names; if none is found, set `TEXT_COL` manually.


In [ ]:
candidate_text_columns = [
    "comment", "text", "comment_text", "Comment", "Text", "body"
]

TEXT_COL = next((c for c in candidate_text_columns if c in df.columns), None)

if TEXT_COL is None:
    raise ValueError(
        "No comment-text column detected. Set TEXT_COL manually."
    )

print("Text column:", TEXT_COL)


## 2. Data cleaning and linguistic preprocessing

In [ ]:
# Remove missing/empty comments and exact duplicates.
df = df.copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

df = df[df[TEXT_COL].ne("")]
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Clean corpus size:", len(df))


In [ ]:
import spacy

nlp = spacy.load("it_core_news_lg")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^\wÀ-ÿ'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["normalized_text"] = df[TEXT_COL].map(clean_text)

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join(
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and token.lemma_.strip()
    )

df["clean_text"] = df["normalized_text"].map(lemmatize_text)

df[["clean_text"]].head()


## 3. Exploratory Data Analysis

In [ ]:
df["word_count"] = df[TEXT_COL].str.split().str.len()
df["char_count"] = df[TEXT_COL].str.len()

print(df[["word_count", "char_count"]].describe())
print("\nMedian words:", df["word_count"].median())
print("Mean words:", round(df["word_count"].mean(), 2))
print("Maximum words:", df["word_count"].max())


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["word_count"], bins=30)
plt.xlabel("Number of words")
plt.ylabel("Number of comments")
plt.title("Distribution of comment length")
plt.show()


In [ ]:
from collections import Counter

tokens = " ".join(df["clean_text"]).split()
freq = Counter(tokens)

top30 = pd.DataFrame(freq.most_common(30), columns=["word", "frequency"])
display(top30)

plt.figure(figsize=(12, 6))
plt.bar(top30["word"], top30["frequency"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Frequency")
plt.title("Top 30 lexical items")
plt.tight_layout()
plt.show()


## 4. Transformer-based sentiment analysis

In [ ]:
from transformers import pipeline

# Feel-IT model.
# The exact Hugging Face model identifier may depend on the version
# used in the original Colab environment. Replace MODEL_NAME if necessary.

MODEL_NAME = "MilaNLProc/feel-it-italian-sentiment"

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    truncation=True,
    max_length=512
)


In [ ]:
# Apply the Transformer to the original comment text.
# For large datasets, batch processing can be used to improve performance.

results = sentiment_pipeline(
    df[TEXT_COL].tolist(),
    batch_size=16
)

df["transformer_label_raw"] = [r["label"] for r in results]
df["transformer_confidence"] = [r["score"] for r in results]

label_map = {
    "POSITIVE": "Positive",
    "NEGATIVE": "Negative",
    "NEUTRAL": "Neutral",
    "positive": "Positive",
    "negative": "Negative",
    "neutral": "Neutral",
}

df["sentiment"] = df["transformer_label_raw"].map(
    lambda x: label_map.get(str(x), str(x).title())
)

display(df[[TEXT_COL, "sentiment", "transformer_confidence"]].head())


In [ ]:
sentiment_distribution = (
    df["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("percentage")
)

display(sentiment_distribution)

plt.figure(figsize=(7, 5))
sentiment_distribution.plot(kind="bar")
plt.ylabel("Percentage")
plt.xlabel("Sentiment")
plt.title("Sentiment distribution")
plt.xticks(rotation=0)
plt.show()


## 5. Supervised Machine Learning: TF-IDF + Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

X_text = df["clean_text"]
y = df["sentiment"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=3000
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Training matrix:", X_train.shape)
print("Test matrix:", X_test.shape)


In [ ]:
clf = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
print()
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
labels = ["Negative", "Neutral", "Positive"]

cm = confusion_matrix(y_test, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot()
plt.title("Logistic Regression — Confusion Matrix")
plt.show()


## 6. Feature interpretation

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

for class_name, class_coef in zip(clf.classes_, clf.coef_):
    top_positive = feature_names[np.argsort(class_coef)[-10:][::-1]]
    print(f"\n{class_name}:")
    print(", ".join(top_positive))


In [ ]:
# More explicit coefficient table
feature_tables = {}

for class_name, class_coef in zip(clf.classes_, clf.coef_):
    idx = np.argsort(class_coef)[-10:][::-1]
    feature_tables[class_name] = pd.DataFrame({
        "feature": feature_names[idx],
        "coefficient": class_coef[idx]
    })

for class_name, table in feature_tables.items():
    print(f"\nTop features — {class_name}")
    display(table)


## 7. Unsupervised Machine Learning: K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# The project reports that the final clustering uses unigrams.
cluster_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    max_features=3000
)

X_cluster = cluster_vectorizer.fit_transform(df["clean_text"])

print("Clustering matrix:", X_cluster.shape)


In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    labels_k = model.fit_predict(X_cluster)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels_k))

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouettes, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.show()


In [ ]:
K = 6

kmeans = KMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X_cluster)

cluster_distribution = (
    df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .to_frame("comments")
)

cluster_distribution["percentage"] = (
    cluster_distribution["comments"] / len(df) * 100
).round(1)

display(cluster_distribution)


## 8. Cluster visualisation with PCA

In [ ]:
from sklearn.decomposition import PCA

# Convert the sparse TF-IDF matrix to dense form for PCA.
X_dense = X_cluster.toarray()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_dense)

df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

plt.figure(figsize=(9, 7))

for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]
    plt.scatter(
        subset["pca_1"],
        subset["pca_2"],
        label=f"Cluster {cluster_id}",
        alpha=0.6
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA projection of K-Means clusters")
plt.legend()
plt.show()


## 9. Cluster-level lexical interpretation

In [ ]:
cluster_terms = {}

cluster_feature_names = np.array(cluster_vectorizer.get_feature_names_out())

for cluster_id in range(K):
    centroid = kmeans.cluster_centers_[cluster_id]
    top_idx = np.argsort(centroid)[-15:][::-1]
    cluster_terms[cluster_id] = cluster_feature_names[top_idx]

for cluster_id, terms in cluster_terms.items():
    print(f"Cluster {cluster_id}: {', '.join(terms)}")


## 10. Linguistic interpretation

The quantitative outputs should not be treated as self-explanatory.

The project specifically investigates the distinction between:

- emotional polarity and evaluation of the discourse object;
- lexical signals and contextual meaning;
- computational classification and pragmatic interpretation.

Particular attention should be paid to comments involving irony, implicit meaning, metaphor, hyperbole, cultural references, nostalgia, shared narrative knowledge and indirect evaluation.

For example, words associated with fear, death or sadness may trigger a negative classification even when the broader comment expresses enthusiasm or emotional attachment to the series.

The clustering results should likewise be interpreted as exploratory lexical groupings rather than as definitive semantic categories.


## 11. Export processed data and results

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

processed_path = RESULTS_DIR / "processed_comments.csv"
df.to_csv(processed_path, index=False)

cluster_distribution.to_csv(
    RESULTS_DIR / "cluster_distribution.csv"
)

sentiment_distribution.to_csv(
    RESULTS_DIR / "sentiment_distribution.csv"
)

print("Saved:", processed_path)


## 12. Reproducibility notes

The original project was developed in Python 3 and Google Colab.

For local execution:

```bash
pip install -r requirements.txt
python -m spacy download it_core_news_lg
```

Then open this notebook and place the raw dataset at:

```text
data/raw/comments.csv
```

The repository intentionally separates raw data from processed outputs. Public datasets should only be redistributed when their use and redistribution are appropriate.

### Analytical caution

The Logistic Regression model uses Transformer-generated labels as its target. Its accuracy therefore measures how well a TF-IDF linear classifier reproduces the Transformer decisions; it is **not** a measure of agreement with human gold-standard sentiment annotations.

Likewise, `k=6` is treated as an operational choice for exploratory analysis, not as evidence that six naturally occurring semantic categories exist in the corpus.
